# Deepfake Detection — Exploratory Data Analysis

Dataset: [Deepfake and Real Images](https://www.kaggle.com/datasets/manjilkarki/deepfake-and-real-images)

Run this notebook on **Google Colab** — instructions in cell 1.

In [ ]:
# ── COLAB SETUP (run this cell first on Colab) ──────────────
import os, sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive')

    # Set this to wherever you uploaded the project on Drive
    PROJECT_DIR = '/content/drive/MyDrive/deepfake-detection'
    os.chdir(PROJECT_DIR)
    sys.path.insert(0, PROJECT_DIR)

    # Install dependencies
    os.system('pip install -q timm albumentations pytorch-grad-cam facenet-pytorch')
    print('Colab setup complete ✓')
else:
    # Local — go to project root
    os.chdir('..')

print(f'Working directory: {os.getcwd()}')

In [ ]:
import yaml, cv2, random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.notebook import tqdm
from src.dataset import collect_image_paths, load_config, get_train_transforms

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size']  = 11

cfg = load_config('config.yaml')
d   = cfg['data']
print('Config loaded ✓')

## 1. Dataset Overview

In [ ]:
splits = {
    'Train': (d['train_real'], d['train_fake']),
    'Val':   (d['val_real'],   d['val_fake']),
    'Test':  (d['test_real'],  d['test_fake']),
}

stats = {}
for split, (rdir, fdir) in splits.items():
    r = collect_image_paths(rdir)
    f = collect_image_paths(fdir)
    stats[split] = {'real': len(r), 'fake': len(f), 'real_paths': r, 'fake_paths': f}
    print(f'{split:<6} → Real: {len(r):>7,}  Fake: {len(f):>7,}  Total: {len(r)+len(f):>8,}')

total_real = sum(v['real'] for v in stats.values())
total_fake = sum(v['fake'] for v in stats.values())
print(f'\nGrand total → Real: {total_real:,}  Fake: {total_fake:,}  ({total_real+total_fake:,} images)')

In [ ]:
os.makedirs('results/plots', exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Dataset Overview', fontsize=15, fontweight='bold')

# Stacked bar — split breakdown
split_names = list(stats.keys())
reals = [stats[s]['real'] for s in split_names]
fakes = [stats[s]['fake'] for s in split_names]
x = range(len(split_names))
b1 = axes[0].bar(x, reals, color='#2E86AB', label='Real')
b2 = axes[0].bar(x, fakes, bottom=reals, color='#E84855', label='Fake')
axes[0].set_xticks(x); axes[0].set_xticklabels(split_names, fontsize=12)
axes[0].set_ylabel('Images'); axes[0].set_title('Split Distribution')
axes[0].legend(); axes[0].spines[['top','right']].set_visible(False)
for bar, val in zip(b1, reals):
    axes[0].text(bar.get_x()+bar.get_width()/2, val/2, f'{val:,}', ha='center', va='center', color='white', fontsize=9, fontweight='bold')

# Pie — overall
axes[1].pie([total_real, total_fake], labels=['Real','Fake'],
            colors=['#2E86AB','#E84855'], autopct='%1.1f%%',
            startangle=90, textprops={'fontsize':13})
axes[1].set_title('Overall Class Balance')

plt.tight_layout()
plt.savefig('results/plots/class_distribution.png', bbox_inches='tight')
plt.show()
print('Saved → results/plots/class_distribution.png')

## 2. Sample Images — Real vs Fake

In [ ]:
real_paths = stats['Train']['real_paths']
fake_paths = stats['Train']['fake_paths']

n = 8
real_sample = random.sample(real_paths, n)
fake_sample = random.sample(fake_paths, n)

fig, axes = plt.subplots(2, n, figsize=(n*2.2, 5))
fig.suptitle('Training Samples — Real (top) vs Fake (bottom)', fontsize=13, fontweight='bold')

for i in range(n):
    for row, path, color in [(0, real_sample[i], '#2E86AB'), (1, fake_sample[i], '#E84855')]:
        img = cv2.imread(str(path))
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            axes[row, i].imshow(img)
        axes[row, i].axis('off')
        if i == 0:
            axes[row, i].set_title(['REAL','FAKE'][row], fontsize=11,
                                    fontweight='bold', color=color)

plt.tight_layout()
plt.savefig('results/plots/sample_images.png', bbox_inches='tight')
plt.show()

## 3. Image Statistics (Brightness, Contrast, Resolution)

In [ ]:
def compute_stats(paths, n_sample=300):
    sample = random.sample(paths, min(n_sample, len(paths)))
    brightness, contrast, heights, widths = [], [], [], []
    for p in tqdm(sample, leave=False):
        img = cv2.imread(str(p))
        if img is None: continue
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        brightness.append(float(gray.mean()))
        contrast.append(float(gray.std()))
        heights.append(img.shape[0])
        widths.append(img.shape[1])
    return dict(brightness=brightness, contrast=contrast, height=heights, width=widths)

print('Computing stats on 300 samples each...')
rs = compute_stats(real_paths)
fs = compute_stats(fake_paths)

print(f"Real → Brightness: {np.mean(rs['brightness']):.1f} ± {np.std(rs['brightness']):.1f}  "
      f"Contrast: {np.mean(rs['contrast']):.1f}")
print(f"Fake → Brightness: {np.mean(fs['brightness']):.1f} ± {np.std(fs['brightness']):.1f}  "
      f"Contrast: {np.mean(fs['contrast']):.1f}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('Image Statistics — Real vs Fake', fontsize=14, fontweight='bold')

for ax, key, label in [
    (axes[0,0], 'brightness', 'Brightness (mean pixel)'),
    (axes[0,1], 'contrast',   'Contrast (pixel std)'),
    (axes[1,0], 'height',     'Image Height (px)'),
    (axes[1,1], 'width',      'Image Width (px)'),
]:
    ax.hist(rs[key], bins=40, alpha=0.7, color='#2E86AB', label='Real', density=True)
    ax.hist(fs[key], bins=40, alpha=0.7, color='#E84855', label='Fake', density=True)
    ax.set_xlabel(label); ax.set_ylabel('Density')
    ax.set_title(label, fontweight='bold'); ax.legend()
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('results/plots/image_statistics.png', bbox_inches='tight')
plt.show()

## 4. Frequency Domain Analysis (FFT)
GAN-generated deepfakes often leave artifacts in the frequency domain invisible to the human eye.

In [ ]:
def avg_fft(paths, n=150):
    sample = random.sample(paths, min(n, len(paths)))
    acc, count = None, 0
    for p in tqdm(sample, leave=False):
        img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        if img is None: continue
        img = cv2.resize(img, (256, 256)).astype(np.float32)
        mag = np.log(np.abs(np.fft.fftshift(np.fft.fft2(img))) + 1)
        acc = mag if acc is None else acc + mag
        count += 1
    return acc / count if count else np.zeros((256,256))

print('Computing FFT (this takes ~30s)...')
real_fft = avg_fft(real_paths)
fake_fft = avg_fft(fake_paths)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Average FFT Spectrum — Real vs Fake', fontsize=14, fontweight='bold')

axes[0].imshow(real_fft, cmap='hot'); axes[0].set_title('Real'); axes[0].axis('off')
axes[1].imshow(fake_fft, cmap='hot'); axes[1].set_title('Fake'); axes[1].axis('off')
diff = np.abs(fake_fft - real_fft)
im = axes[2].imshow(diff, cmap='RdYlGn_r')
axes[2].set_title('|Fake − Real|  (frequency artifacts)')
axes[2].axis('off')
plt.colorbar(im, ax=axes[2], fraction=0.046)

plt.tight_layout()
plt.savefig('results/plots/fft_analysis.png', bbox_inches='tight')
plt.show()
print('Bright regions in the difference map = frequency artifacts unique to deepfakes.')

## 5. Augmentation Preview

In [ ]:
tf = get_train_transforms(cfg)
MEAN = np.array(cfg['augmentation']['normalize_mean'])
STD  = np.array(cfg['augmentation']['normalize_std'])

img_path = random.choice(real_paths)
img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.suptitle('8 Augmentation Variants of the Same Image', fontsize=13, fontweight='bold')
for i, ax in enumerate(axes.flat):
    t = tf(image=img)['image'].numpy().transpose(1,2,0)
    ax.imshow(np.clip(t*STD + MEAN, 0, 1))
    ax.set_title(f'Variant {i+1}', fontsize=9); ax.axis('off')

plt.tight_layout()
plt.savefig('results/plots/augmentation_preview.png', bbox_inches='tight')
plt.show()

## 6. Summary

In [ ]:
print('='*55)
print('  EDA COMPLETE — all plots saved to results/plots/')
print('='*55)
print(f"  Train → {stats['Train']['real']:,} real + {stats['Train']['fake']:,} fake")
print(f"  Val   → {stats['Val']['real']:,} real + {stats['Val']['fake']:,} fake")
print(f"  Test  → {stats['Test']['real']:,} real + {stats['Test']['fake']:,} fake")
print()
print('  Next step: run src/train.py on Colab!')